In [1]:
#Nuraleya Binti Azizan (IS01084475)
import pandas as pd
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report

In [2]:
# Load dataset
df = pd.read_csv("Reviews.csv")

# Display first 5 rows
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [3]:
# Keep only necessary columns
df = df[['Score', 'Text']]

# Convert score to sentiment
def get_sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df['Sentiment'] = df['Score'].apply(get_sentiment)

# Drop missing values
df.dropna(inplace=True)

# Take smaller sample (important - dataset very large)
df = df.sample(5000, random_state=42)

In [4]:
# Lexicon based approach
analyzer = SentimentIntensityAnalyzer()

# Store results
tb_predictions = []
vader_predictions = []
actual_labels = df['Sentiment'].tolist()

for text in df['Text']:
    
    # TextBlob
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity
    
    if polarity > 0:
        tb_predictions.append('positive')
    elif polarity < 0:
        tb_predictions.append('negative')
    else:
        tb_predictions.append('neutral')
    
    # VADER
    score = analyzer.polarity_scores(text)['compound']
    
    if score > 0.05:
        vader_predictions.append('positive')
    elif score < -0.05:
        vader_predictions.append('negative')
    else:
        vader_predictions.append('neutral')

In [5]:
print("TextBlob Classification Report:")
print(classification_report(actual_labels, tb_predictions))

print("\nVADER Classification Report:")
print(classification_report(actual_labels, vader_predictions))

TextBlob Classification Report:
              precision    recall  f1-score   support

    negative       0.52      0.39      0.45       703
     neutral       0.17      0.03      0.05       378
    positive       0.84      0.94      0.89      3919

    accuracy                           0.80      5000
   macro avg       0.51      0.45      0.46      5000
weighted avg       0.74      0.80      0.76      5000


VADER Classification Report:
              precision    recall  f1-score   support

    negative       0.59      0.40      0.48       703
     neutral       0.15      0.04      0.06       378
    positive       0.85      0.96      0.90      3919

    accuracy                           0.81      5000
   macro avg       0.53      0.46      0.48      5000
weighted avg       0.76      0.81      0.78      5000



In [9]:
#ML Approach
texts = df['Text']
labels = df['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.4, random_state=42
)

In [10]:
# Bag of Words
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

# TF-IDF
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [11]:
# Initialize models
nb = MultinomialNB()
svm = SVC(kernel='linear')

# Train on BoW
nb.fit(X_train_bow, y_train)
svm.fit(X_train_bow, y_train)

,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [12]:
# Predictions
nb_pred = nb.predict(X_test_bow)
svm_pred = svm.predict(X_test_bow)

print("\nNaive Bayes Report (BoW):")
print(classification_report(y_test, nb_pred))

print("\nSVM Report (BoW):")
print(classification_report(y_test, svm_pred))


Naive Bayes Report (BoW):
              precision    recall  f1-score   support

    negative       0.87      0.21      0.34       270
     neutral       1.00      0.02      0.04       153
    positive       0.81      0.99      0.89      1577

    accuracy                           0.81      2000
   macro avg       0.89      0.41      0.43      2000
weighted avg       0.83      0.81      0.75      2000


SVM Report (BoW):
              precision    recall  f1-score   support

    negative       0.54      0.53      0.54       270
     neutral       0.26      0.23      0.24       153
    positive       0.88      0.89      0.89      1577

    accuracy                           0.79      2000
   macro avg       0.56      0.55      0.56      2000
weighted avg       0.79      0.79      0.79      2000



In [ ]:
1. TextBlob (Lexicon-Based Sentiment Analysis)

Strengths

Very easy to implement because it does not require training data.
Uses a predefined sentiment lexicon to calculate polarity scores.
Works reasonably well for clear positive reviews, such as reviews containing words like “great”, “amazing”, “delicious”.

Weaknesses

Cannot learn patterns from the dataset because it relies on fixed sentiment dictionaries.
Performs poorly when dealing with long and detailed reviews that contain mixed opinions.
Struggles with neutral sentiment classification.
Does not handle domain-specific product review language effectively.
    
2. VADER (Lexicon-Based Sentiment Analysis)

Strengths

Designed specifically for sentiment analysis in text.
Considers punctuation, capitalization, and sentiment intensity, improving sentiment detection.
Performs well in detecting positive sentiments, especially in expressive reviews.

Weaknesses

Still limited by its predefined lexicon and rules.
Less effective for product-specific vocabulary used in Amazon reviews.
Similar to TextBlob, it struggles with neutral sentiment detection and complex contextual meaning.

3. Naive Bayes (Machine Learning – Bag-of-Words)

Strengths

Works efficiently with large text datasets like the Amazon food reviews dataset.
Performs well for sentiment classification when clear positive or negative words appear in the review text (e.g., “great”, “delicious”, “love”, “excellent”).
Computationally simple and fast to train, making it suitable for large-scale review datasets.
Achieved relatively high overall accuracy and performed very well in identifying positive reviews, which dominate the dataset.

Weaknesses

Assumes that words are independent of each other, which is not realistic in natural language.
Cannot fully understand context or sentence structure in longer reviews.
Performed poorly in detecting neutral and negative sentiments, likely due to class imbalance in the dataset where positive reviews are more frequent.
Words like “not good” may be misinterpreted because the model treats “not” and “good” separately.
    
4. Support Vector Machine (SVM)

Strengths

Works well with high-dimensional text features, especially when using Bag-of-Words or TF-IDF.
Capable of creating a clear boundary between sentiment classes.
Produced more balanced predictions across classes compared to Naive Bayes.
Better at detecting negative reviews, which often contain stronger sentiment expressions.

Weaknesses

Slightly lower overall accuracy in this experiment.
Training time is longer and computational cost is higher compared to Naive Bayes.
Requires parameter tuning and feature selection to achieve optimal performance.
    

    
Overall Observations
The dataset contains many more positive reviews than negative or neutral ones, which affects the performance of all models.
Machine learning models (Naive Bayes and SVM) generally perform better because they learn patterns directly from the dataset.
Lexicon-based models (TextBlob and VADER) are simpler and faster but cannot adapt to domain-specific language or contextual sentiment.
Long review texts, sarcasm, and mixed opinions in product reviews make sentiment classification more challenging.